In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier

from sklearn.preprocessing import MinMaxScaler

import time
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[1]))

from art.attacks.evasion import FastGradientMethod
from utils.functions import run_simple_full_attack, test_simple_model, create_simple_dataset
from utils.models import SimpleNet

/opt/anaconda3/envs/reu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Step 0: Define constants
data_file = "../../data/RandomPos_0709.csv"
bn_model_filename = "RandomPos-full"
adv_model_filename = "RandomPos-adv-train"
eps=0.04

In [ ]:
(x_train, y_train), (x_test, y_test) = create_simple_dataset(data_file=data_file, divide_by=100)

dataset length: 31600
% of attackers in training dataset: 0.2650316455696203
% of attackers in testing dataset: 0.20569620253164558


In [5]:
# Step 2: adversarial train model
# Run a simple adversarial ART attack, end to end. model_filename = None to not save the model
def adv_train(data_file, divide_by, bn_model_filename, adv_model_filename, Attack, **attack_kwargs):

    ## Step 2: Create the model
    model = SimpleNet()

    # Step 2a: Define the loss function and the optimizer

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    ## Step 3: Create the ART classifier

    classifier = PyTorchClassifier(
        model=model,
        clip_values=(x_test.min(), x_test.max()),
        loss=criterion,
        optimizer=optimizer,
        input_shape=(11,),
        nb_classes=2,
    )

    start = time.time()
    # Step 4: Load model
    model.load_state_dict(torch.load(f"../../saved_models/{bn_model_filename}.pth", weights_only=True))
    model.eval()

    # Step 6: Generate adversarial test examples
    attack = Attack(classifier, **attack_kwargs)

    x_train_adv = attack.generate(x=x_train)
    return x_train_adv



In [6]:
x_train_adv = adv_train(data_file=data_file,
          divide_by=2,
          bn_model_filename=bn_model_filename,
          adv_model_filename=adv_model_filename,
          Attack=FastGradientMethod,
          eps=eps)